# 3. A Rede de Monitoramento Brasileira


<div style="text-align: justify">Foram contabilizadas 577 estações de monitoramento da qualidade do ar no Brasil (ano base 2024), representando um acréscimo de 98 estações em relação ao relatório anterior (ano base 2023). Deste montante, 371 são estações que utilizam método de referência ou equivalente e 197 são estações indicativas. Em 9 estações, o método adotado não foi declarado pela UF. Esse acréscimo de unidades de monitoramento está associado ao maior número de respostas das UFs ao questionário aplicado pelo MMA e implantação de mais estações de monitoramento indicativas. </div>

<div style="text-align: justify">Os números reportados acima referem-se às instalações físicas com infraestrutura para abrigar os equipamentos de medição das concentrações de poluentes. Vale ressaltar que uma mesma estação de monitoramento pode medir, simultaneamente, múltiplos poluentes. </div>

<div style="text-align: justify">Neste caso, nas seguintes seções do relatório, alguns resultados são organizados e discutidos de acordo com o número de pontos de monitoramento disponíveis para cada poluente específico. </div>

```{note}
É importante destacar que as estações indicativas são integradas em plataformas internacionais com finalidade científica, exploratória e informativa. A gestão dos dados ocorre através da integração dos equipamentos na plataforma. Em alguns casos, a operação e supervisão é realizada pelos OEMA. 
Estas estações são equipadas com instrumentos e sensores não considerados equivalentes às estações de referência. As estações indicativas são capazes de monitorar a concentração de alguns poluentes atmosféricos em tempo real, no entanto, podem apresentar um grau de incerteza relevante em relação ao dado gerado, principalmente quando não são calibradas e operadas adequadamente.
```

<div style="text-align: justify">É importante ressaltar que, as estações indicativas não atendem aos critérios estabelecidos pelo Guia Técnico para o Monitoramento e Avaliação da Qualidade do Ar <cite id="ip3dh"><a href="#zotero%7C22267313%2FM8BLF2PV">(BRASIL, 2020)</a></cite>, mas podem fornecer informações relevantes sobre a qualidade do ar, principalmente em locais sem nenhuma estação de referência ou equivalente.</div><br/>

```{note}
As estações sem registro de início e fim de operação foram contabilizadas apenas nos anos de 2023 e 2024. Alguns períodos de operação foram estimados a partir da data de instalação e operação.
```

<div style="text-align: justify">As Figuras 20, 21 e 22 mostram, respectivamente, a evolução do número de estações de monitoramento em cada UF no Brasil, a expansão ou retração da rede estadual em cada ano, e o número de pontos de monitoramento por poluente que foram registrados na base de dados desenvolvida neste relatório.</div><br/>

In [1]:
# -*- coding: utf-8 -*-
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os
from IPython.display import HTML

# ===========================
# Caminhos
# ===========================
rootPath = os.path.dirname(os.getcwd())
aqm_path = os.path.join(rootPath, "data", "Monitoramento_QAr_BR.csv")

# ===========================
# Leitura e pré-processamento
# ===========================
aqmData = pd.read_csv(aqm_path)

# Caso a coluna ANOS_MONITORADOS seja string tipo "2019,2020"
if aqmData["ANOS_MONITORADOS"].dtype == object:
    aqmData["ANOS_MONITORADOS"] = aqmData["ANOS_MONITORADOS"].apply(
        lambda x: [int(y) for y in str(x)
                   .replace("[","")
                   .replace("]","")
                   .replace("'", "")
                   .split(",")
                   if y.strip().isdigit()]
    )

# Explode anos monitorados
df_exploded = aqmData.explode("ANOS_MONITORADOS")

# Consolidar anos por UF e ID_OEMA
aqmData_consolidated = (
    df_exploded.groupby(['UF','ID_OEMA'])['ANOS_MONITORADOS']
    .apply(lambda x: ', '.join(sorted(set(map(str, x)))))
    .reset_index()
)

# Remover nulos
aqmData_consolidated.loc[
    aqmData_consolidated['ANOS_MONITORADOS'] == 'nan', 'ANOS_MONITORADOS'
] = np.nan
aqmData_consolidated = aqmData_consolidated.dropna(subset=['ANOS_MONITORADOS'])

# Converter string → lista de inteiros
def process_years(x):
    if pd.isna(x):
        return []
    years = [int(y.strip()) for y in str(x).split(',') if y.strip().isdigit()]
    return sorted(set(years))

aqmData_consolidated['ANOS_MONITORADOS'] = aqmData_consolidated['ANOS_MONITORADOS'].apply(process_years)

# Explode novamente
df_exploded_years = aqmData_consolidated.explode('ANOS_MONITORADOS')
df_exploded_years = df_exploded_years.dropna(subset=['ANOS_MONITORADOS'])
df_exploded_years['ANOS_MONITORADOS'] = df_exploded_years['ANOS_MONITORADOS'].astype(int)

# ===========================
# Contagem de estações por UF e ano
# ===========================
df_counts = (
    df_exploded_years.groupby(['ANOS_MONITORADOS', 'UF'])['ID_OEMA']
    .nunique()
    .reset_index(name='Num_Stations')
)
df_counts = df_counts[df_counts['ANOS_MONITORADOS'] <= 2024]

anos = sorted(df_counts['ANOS_MONITORADOS'].unique())
ufs = sorted(df_counts['UF'].unique())

# ===========================
# Figura: Barras empilhadas
# ===========================
layout_style = dict(
    plot_bgcolor="whitesmoke",
    paper_bgcolor="white",
    hovermode='x unified',
    legend_title="UF",
    height=600
)

fig_bars = go.Figure()
for uf in ufs:
    df_uf = df_counts[df_counts['UF'] == uf].sort_values('ANOS_MONITORADOS')
    fig_bars.add_trace(go.Bar(
        x=df_uf['ANOS_MONITORADOS'],
        y=df_uf['Num_Stations'],
        name=uf
    ))

fig_bars.update_layout(
    **layout_style,
    barmode='stack',
    xaxis_title="Ano",
    yaxis_title="Número de pontos de monitoramento",
    xaxis_range=[df_exploded_years['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    xaxis=dict(
        type='linear',
        tickmode='array',
        tickvals=anos,
        ticktext=anos
    )
)

# ===========================
# Exportar para HTML
# ===========================
html_path = os.path.join(rootPath, "_static", "monitoramento", "fig_num_pontos_UF.html")
os.makedirs(os.path.dirname(html_path), exist_ok=True)
fig_bars.write_html(html_path, include_plotlyjs="cdn")

HTML(fig_bars.to_html(include_plotlyjs="cdn"))


<p style="text-align:center; color:#636D7D; font-size:15.5px;">
  <em>Fig. 20</em> Número de estações de monitoramento por UF.
</p>

```{tip}
Escolha um estado na barra lateral à direita para filtrar a figura e visualizar seus dados ao longo dos anos.
```

In [2]:
# ---------------------------
# Figure 2: Difference lines
# ---------------------------
layout_style = dict(
    plot_bgcolor="white",   # light gray for plot area
    paper_bgcolor="white",  # light gray for outer background
    hovermode='x unified',
    legend_title="UF",
    height=800
)

fig_diff = go.Figure()
difUFmin = []
difUFmax = []
for uf in ufs:
    df_uf = df_counts[df_counts['UF'] == uf].sort_values('ANOS_MONITORADOS')
    df_uf['Difference'] = df_uf['Num_Stations'].diff().fillna(0)
    fig_diff.add_trace(go.Scatter(
        x=df_uf['ANOS_MONITORADOS'],
        y=df_uf['Difference'],
        mode='lines+markers',
        name=uf
    ))
    difUFmin.append(df_uf['Difference'].min())
    difUFmax.append(df_uf['Difference'].max())

max_abs_value = max(max(difUFmin, key=abs),max(difUFmax, key=abs))

fig_diff.update_layout(
    **layout_style,
    #title="Diferença anual de pontos de monitoramento por UF",
    xaxis_title="Ano",
    yaxis_title="Diferença anual",
    #legend_title="UF",
    #hovermode='x unified',
    #height=600,
    xaxis_range=[df_exploded_years['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
    shapes=[
        dict(
            type="rect",
            xref="paper", x0=0, x1=1,         # full width
            yref="y", y0=-max_abs_value-1, y1=0,  
            fillcolor="mistyrose",            # light red background
            opacity=0.5,
            layer="below",
            line_width=0
        ),
        dict(
            type="rect",
            xref="paper", x0=0, x1=1,         # full width
            yref="y", y0=max_abs_value+1, y1=0,  
            fillcolor="lightblue",            # light red background
            opacity=0.5,
            layer="below",
            line_width=0
        ),
        # Add thick horizontal line at y=0
        dict(
            type="line",
            xref="paper", x0=0, x1=1,  # full width of the plot
            yref="y", y0=0, y1=0,
            line=dict(color="white", width=6),  # thicker line
            layer="below"               # 👈 puts it behind all data
        )
    ]
)

# Display figures

#fig_diff.show()
HTML(fig_diff.to_html(include_plotlyjs="cdn"))

<p style="text-align:center; color:#636D7D; font-size:15.5px;">
  <em>Fig. 21</em> Evolução anual do número de estações de monitoramento por UF.
</p>

```{tip}
Selecione o poluente no dropdown menu e avalie a evolução do número de pontos de monitoramento. Clique na legenda para controlar os estados que agregam estações na figura.   
```

In [3]:
# Caminho para a pasta de dados
rootPath = os.path.dirname(os.getcwd())

# Lendo o csv
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')


df = aqmData.copy()

# Quebrar ANOS_MONITORADOS em listas
df["ANOS_MONITORADOS"] = df["ANOS_MONITORADOS"].str.split(",")

# To remove all spaces
df['UF'] = df['UF'].str.replace(" ", "")


# Explodir para cada ano virar uma linha
df_exploded = df.explode("ANOS_MONITORADOS")

# Converter para inteiro
#df_exploded["ANOS_MONITORADOS"] = df_exploded["ANOS_MONITORADOS"].astype(int)

# Agrupar também por POLUENTE
df_grouped = (
    df_exploded.groupby(["POLUENTE", "UF", "ANOS_MONITORADOS"])
    .size()   # conta linhas
    .reset_index(name="NSTATION")
)
# Remover linhas com valores não numéricos
df_grouped = df_grouped[pd.to_numeric(df_grouped['ANOS_MONITORADOS'], errors='coerce').notna()]

# Converter agora para inteiro
df_grouped['ANOS_MONITORADOS'] = df_grouped['ANOS_MONITORADOS'].astype(int)





# Lista de poluentes e UFs
poluentes = sorted(df_grouped["POLUENTE"].unique())
ufs = sorted(df_grouped["UF"].unique())


# Cores fixas por UF
color_map = {uf: f"hsl({i*40 % 360},70%,50%)" for i, uf in enumerate(ufs)}

layout_style = dict(
    plot_bgcolor="whitesmoke",   # light gray for plot area
    paper_bgcolor="white",  # light gray for outer background
    hovermode='x unified',
    legend_title="UF",
    height=800
)
fig = go.Figure()
trace_visibility = []

## Track UF que já teve legenda
#uf_legend_shown = {uf: False for uf in ufs}

# Criar todos os traços: um trace por UF e poluente
for pol in poluentes:
    df_pol = df_grouped[df_grouped["POLUENTE"] == pol]
    for uf in ufs:
        df_uf = df_pol[df_pol["UF"] == uf]
        y_values = df_uf["NSTATION"].values
        x_values = df_uf["ANOS_MONITORADOS"].values 

        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                name=uf,
                marker=dict(color=color_map[uf]),
                legendgroup=uf,
                showlegend=True,  # apenas um trace de cada UF aparece na legenda
                visible=(pol == poluentes[0]),  # primeiro poluente aparece inicialmente
                text=y_values,
                textposition="outside"
            )
        )
        #uf_legend_shown[uf] = True
        trace_visibility.append(pol)

# Dropdown menu
buttons = []
for pol in poluentes:
    vis = [p == pol for p in trace_visibility]
    buttons.append(dict(
        label=pol,
        method="update",
        args=[{"visible": vis},
              {"title": f"Monitoramento de {pol}"}]
    ))
anos = np.arange(df_grouped['ANOS_MONITORADOS'].min(),
                 df_grouped['ANOS_MONITORADOS'].max() + 1).tolist()

# Layout
fig.update_layout(
    **layout_style,
    #height=600,
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-.4,
        xanchor="left",
        x=0,
        title="UF: "
    ),
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
    xaxis_range=[df_grouped['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    yaxis=dict(
        type='linear',
        title="N. pontos de monitoramento",
        tickmode='array',
    ),
    margin=dict(r=120),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.15,
        y=1.05,
        showactive=True,
        
    )],
    title=f"Monitoramento de {poluentes[0]}"
)

HTML(fig.to_html(include_plotlyjs="cdn"))


<p style="text-align:center; color:#636D7D; font-size:15.5px;">
  <em>Fig. 22</em> Número de pontos de monitoramento por poluente e UF. 
</p>

```{note}
O primeiro poluente monitorado e registrado no Brasil foram as Partículas Totais em Suspensão (PTS). Atualmente, o MP2,5 é o poluente monitorado em um maior número de pontos de monitoramento.
```

### Referências

<!-- BIBLIOGRAPHY START -->
<div class="csl-bib-body">
  <div class="csl-entry"><i id="zotero|22267313/M8BLF2PV"></i>BRASIL. <b>Guia Técnico para o Monitoramento e Avaliação da Qualidade do Ar</b>. , 2020. </div>
</div>
<!-- BIBLIOGRAPHY END -->